# Notebook 20 — ERA5 wind and solar capacity-factor pilot

Run each cell in order using the repository **.venv (Python 3.12)** kernel.
This exploratory notebook writes only weather caches and pilot outputs; it does not modify
production Silver/Gold workflows or the model database. The default is **24 hourly samples
on 6 June 2020 over southern Ontario**, not an annual resource assessment.

The earlier one-hour GRIB file tested CDS access. Atlite retrieves its own complete feature
sets here (wind, radiation and temperature). Keep CDS credentials in your home `.cdsapirc`;
never paste them into this notebook. Only cell 4 submits weather requests.

Sources: [Atlite](https://github.com/PyPSA/atlite),
[ERA5 single levels](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels),
[CDS setup](https://cds.climate.copernicus.eu/how-to-api).


## 1. Imports, kernel and repository location
Check the executable below before continuing. No credentials are displayed.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import logging
import sys

import atlite
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from atlite.resource import get_windturbineconfig, get_solarpanelconfig

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').exists() and (p / 'src/geocanoe').is_dir()), None)
assert ROOT is not None, 'Start Jupyter from the repository or its notebooks directory.'
logging.basicConfig(level=logging.INFO)
VERSIONS = {name: version(name) for name in ['atlite', 'cdsapi', 'xarray', 'numpy', 'pandas', 'geopandas']}
print('Python:', sys.executable)
print('Repository:', ROOT)
display(pd.Series(VERSIONS, name='version'))


## 2. Explicit pilot assumptions
These turbine and PV choices demonstrate conversion, rather than representing a calibrated
Canadian fleet. Solar capacity is normalized to the panel model rating; define an AC/DC basis
before using it in TEMOA. No additional wake, availability, snow, icing or electrical-loss
adjustments are applied. Bounds are longitude/latitude in EPSG:4326.

Keep the small defaults for the first run. For a full year, change the dates to January 1
00:00 through December 31 23:00. A leap year has 8,784 hours. Wind Atlas refinement and
MERRA-2 comparison remain future work; interpolation is not physical downscaling.


In [ ]:
CONFIG = {
    'west': -82.0, 'east': -78.0, 'south': 43.0, 'north': 45.0,
    'start': '2020-06-06 00:00', 'end': '2020-06-06 23:00',
    'dx': 0.25, 'dy': 0.25,
    'turbine': 'Vestas_V112_3MW', 'panel': 'CSi',
    'orientation': {'slope': 35.0, 'azimuth': 180.0},
    'wind_interpolation': 'logarithmic',
    'features': ['wind', 'influx', 'temperature'],
}
assert CONFIG['west'] < CONFIG['east'] and CONFIG['south'] < CONFIG['north']
expected_time = pd.date_range(CONFIG['start'], CONFIG['end'], freq='h')
assert len(expected_time) > 0
assert CONFIG['turbine'] in atlite.windturbines
assert CONFIG['panel'] in atlite.solarpanels
identity = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]
CACHE = ROOT / 'data_files/raw/era5/atlite_pilot'
OUT = ROOT / 'data_files/processed/renewables/atlite_pilot' / identity
CACHE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
CUTOUT_PATH = CACHE / f'era5_{identity}.nc'
display(pd.Series(CONFIG, dtype=object))
print('Expected hourly samples:', len(expected_time))
print('Cache:', CUTOUT_PATH)
print('Output:', OUT)
print('Turbine hub height (m):', get_windturbineconfig(CONFIG['turbine'])['hub_height'])
get_solarpanelconfig(CONFIG['panel'])


## 3. Construct or reopen a cutout
This defines coordinates without submitting a CDS request. An identical configuration reuses its cache.


In [ ]:
cutout = atlite.Cutout(
    path=CUTOUT_PATH, module='era5',
    x=slice(CONFIG['west'], CONFIG['east']),
    y=slice(CONFIG['south'], CONFIG['north']),
    time=slice(CONFIG['start'], CONFIG['end']),
    dx=CONFIG['dx'], dy=CONFIG['dy'],
)
print(cutout)
print('Weather grid cells:', len(cutout.grid))
print('Grid-cell hours:', len(cutout.grid) * len(expected_time))


## 4. Retrieve and prepare ERA5 — network cell
Run this cell when ready to submit the small request. CDS credentials and accepted ERA5
terms are required. Atlite requests wind components and roughness, solar radiation fields,
and temperature. Queue time varies. Prepared features are reused on subsequent runs.
This small pilot requests NetCDF directly to avoid the Windows ecCodes DLL dependency.
The prepared cutout is also stored as NetCDF. Larger requests may need smaller spatial or
temporal batches because CDS NetCDF conversion has request-size limits.


In [ ]:
cutout.prepare(
    features=CONFIG['features'], data_format='netcdf',
    monthly_requests=True, concurrent_requests=False,
    show_progress=True, dask_kwargs={'scheduler': 'single-threaded'},
)
display(cutout.data)


## 5. Check weather completeness
Retain ERA5 UTC timestamps. Radiation values describe preceding-hour intervals; Atlite
handles the solar-position timing adjustment. Do not shift the weather timestamps manually.


In [ ]:
required = ['wnd100m', 'roughness', 'influx_direct', 'influx_diffuse', 'temperature']
assert set(required).issubset(cutout.data.data_vars), 'Missing prepared features.'
assert pd.DatetimeIndex(cutout.data.time.values).equals(expected_time), 'Unexpected hourly coverage.'
weather_missing = cutout.data[required].isnull().sum().compute().to_array().to_series()
display(weather_missing.rename('missing_values'))
assert (weather_missing == 0).all(), 'Investigate missing weather before conversion.'


## 6. Wind conversion
Extrapolate wind to the configured turbine hub height and apply its power curve. Preserve every hour and weather cell.


In [ ]:
wind_cf = cutout.wind(
    turbine=CONFIG['turbine'], interpolation_method=CONFIG['wind_interpolation'],
    aggregate_time=None, show_progress=True,
    dask_kwargs={'scheduler': 'single-threaded'},
).rename('wind_cf')
display(wind_cf)


## 7. Solar PV conversion
Fixed tilt, south-facing panels; no tracking. Preserve hourly output on the same grid.


In [ ]:
solar_cf = cutout.pv(
    panel=CONFIG['panel'], orientation=CONFIG['orientation'], tracking=None,
    aggregate_time=None, show_progress=True,
    dask_kwargs={'scheduler': 'single-threaded'},
).rename('solar_cf')
profiles = xr.Dataset({'wind_cf': wind_cf, 'solar_cf': solar_cf})
profiles.attrs.update(timezone='UTC', pilot_identity=identity,
                      configuration=json.dumps(CONFIG, sort_keys=True))
for name in profiles:
    profiles[name].attrs['units'] = 'p.u.'
display(profiles)


## 8. Validate and summarize the sampled period
Do not silently clip or fill invalid results. PV values above one may reflect rating or
model assumptions and are reported for inspection. These are **sample-period means**,
not annual capacity factors. A simple average across weather cells is only a diagnostic;
it is not an area- or capacity-weighted GeoCANOE regional profile.


In [ ]:
checks = []
for name, da in profiles.data_vars.items():
    assert set(da.dims) == {'time', 'y', 'x'}
    values = da.values
    assert np.isfinite(values).all(), f'{name}: non-finite values'
    assert values.min() >= -1e-8, f'{name}: negative generation'
    if name == 'wind_cf':
        assert values.max() <= 1 + 1e-8, 'Wind exceeds rated output'
    checks.append({'technology': name, 'min': float(values.min()),
                   'max': float(values.max()), 'samples_above_one': int((values > 1).sum()),
                   'sample_mean_equal_cell_weight': float(values.mean())})
checks = pd.DataFrame(checks).set_index('technology')
display(checks)
sample_mean = profiles.mean('time')
hourly_diagnostic = profiles.mean(['y', 'x']).to_dataframe()
hourly_diagnostic.index.name = 'time_utc'
display(hourly_diagnostic.head())


## 9. Inspect hourly profiles and mean resource maps


In [ ]:
ax = hourly_diagnostic.plot(figsize=(11, 4), ylabel='Capacity factor (p.u.)',
                            title='Pilot: equal-weather-cell diagnostic means')
ax.set_xlabel('UTC')
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, profiles.data_vars):
    sample_mean[name].plot(ax=ax, vmin=0)
    ax.set_title(f'{name}: sampled-period mean')
plt.tight_layout()
plt.show()


## 10. GeoCANOE polygon crosswalk
The default is the `prov_quant` 25 km basemap, layer `regions`. Set `BASEMAP_PATH = None` to skip. This maps **sample-period mean** weather-cell
CFs onto only the overlapping portion of each region, using intersection area in EPSG:3347.
Coverage is reported so partial regions cannot be mistaken for complete profiles. ERA5 cell
footprints extend half a grid spacing beyond coordinate bounds. A smaller model grid does not
add weather detail. This step neither excludes water nor models eligible land or capacity density.

For future hourly regional profiles, reuse these intersection weights on each hourly CF.
For capacity-weighted profiles, replace area weights with an explicit capacity allocation.


In [ ]:
BASEMAP_PATH = ROOT / 'data_files/processed/basemaps/prov_quant_basemap_25km_centroid.gpkg'
BASEMAP_LAYER = 'regions'
regional_summary = None
if BASEMAP_PATH is None:
    print('Optional crosswalk skipped; set BASEMAP_PATH to enable.')
else:
    regions = gpd.read_file(BASEMAP_PATH, layer=BASEMAP_LAYER)
    assert regions.crs is not None and regions['region'].is_unique
    regions = regions[['region', 'geometry']].to_crs(3347)
    regions['region_area_m2'] = regions.area
    weather = cutout.grid.copy()
    for name in profiles:
        mean = sample_mean[name]
        weather[name] = [float(mean.sel(x=x, y=y)) for x, y in zip(weather.x, weather.y)]
    weather = weather.to_crs(3347)
    pieces = gpd.overlay(regions, weather[['wind_cf', 'solar_cf', 'geometry']], how='intersection')
    if pieces.empty:
        raise ValueError('Basemap does not overlap the ERA5 pilot; choose overlapping bounds.')
    pieces['overlap_m2'] = pieces.area
    pieces = pieces.loc[pieces.overlap_m2 > 0].copy()
    grouped = pieces.groupby('region')
    regional_summary = grouped['overlap_m2'].sum().to_frame('covered_area_m2')
    for name in profiles:
        pieces[f'{name}_weighted'] = pieces[name] * pieces.overlap_m2
        regional_summary[name] = pieces.groupby('region')[f'{name}_weighted'].sum() / regional_summary.covered_area_m2
    regional_summary = regions.set_index('region')[['region_area_m2']].join(regional_summary)
    regional_summary['coverage_fraction'] = regional_summary.covered_area_m2.fillna(0) / regional_summary.region_area_m2
    display(regional_summary)


## 10a. Capacity-factor heat maps over the 25 km basemap
Run section 10 first. The two panels show **sample-period mean** capacity factors,
weighted by overlap area within each GeoCANOE region. Only the covered portions of
regions are coloured; grey context is outside the weather footprint. This avoids extending
the southern Ontario pilot values across the national basemap. The dashed line marks the
ERA5 cell footprint, which extends half a grid spacing beyond the coordinate bounds.

The interactive map has wind and solar layers and region tooltips, including coverage.
Use its layer control to switch technologies. No internet tiles are needed; the GeoCANOE
polygons provide the basemap. HTML is saved alongside the pilot outputs (viewing HTML
may require internet for Folium's JavaScript assets). The PNG is a standalone export.

In [ ]:
if regional_summary is None:
    raise ValueError('Enable BASEMAP_PATH and run section 10 before the preview.')

# Shade intersections only: partial regions retain their coverage warning.
covered_geometry = pieces[['region', 'geometry']].dissolve(by='region')
preview = covered_geometry.join(regional_summary[['wind_cf', 'solar_cf', 'coverage_fraction']])
preview['coverage_percent'] = 100 * preview['coverage_fraction']
footprint = weather[['geometry']].dissolve()
west, south, east, north = footprint.total_bounds
margin = 0.12 * max(east - west, north - south)
view_bounds = (west - margin, south - margin, east + margin, north + margin)
context = regions.cx[view_bounds[0]:view_bounds[2], view_bounds[1]:view_bounds[3]]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, name, label in zip(axes, ['wind_cf', 'solar_cf'], ['Wind', 'Solar PV']):
    context.plot(ax=ax, color='#eeeeee', edgecolor='#aaaaaa', linewidth=0.35)
    preview.plot(column=name, ax=ax, cmap='viridis', vmin=0, vmax=1,
                 edgecolor='#444444', linewidth=0.4, legend=True,
                 legend_kwds={'label': 'Sample-period mean capacity factor (p.u.)', 'shrink': 0.75})
    footprint.boundary.plot(ax=ax, color='#b2182b', linestyle='--', linewidth=1.2)
    ax.set(xlim=view_bounds[::2], ylim=view_bounds[1::2],
           title=f'{label} over prov_quant 25 km regions',
           xlabel='Easting (m), EPSG:3347', ylabel='Northing (m), EPSG:3347')
    ax.ticklabel_format(style='plain', useOffset=False)
fig.suptitle(f"ERA5 pilot: {CONFIG['start']} to {CONFIG['end']} UTC\n"
             'Coloured area = weather-covered portion; grey = no pilot data', fontsize=11)
fig.tight_layout()
fig.savefig(OUT / 'prov_quant_25km_capacity_factor_preview.png', dpi=160, bbox_inches='tight')
plt.show()

import folium
from branca.colormap import linear

preview_wgs = preview.reset_index().to_crs(4326)
context_wgs = context[['region', 'geometry']].to_crs(4326)
footprint_wgs = footprint.to_crs(4326)
interactive_preview = folium.Map(tiles=None, control_scale=True)
folium.GeoJson(
    context_wgs, name='GeoCANOE 25 km context',
    style_function=lambda feature: {'color': '#888888', 'weight': 0.6,
                                    'fillColor': '#eeeeee', 'fillOpacity': 0.4},
).add_to(interactive_preview)
colour_scale = linear.viridis.scale(0, 1)
colour_scale.caption = 'Sample-period mean capacity factor (p.u.)'
for name, label, visible in [('wind_cf', 'Wind capacity factor', True),
                              ('solar_cf', 'Solar PV capacity factor', False)]:
    folium.GeoJson(
        preview_wgs[['region', name, 'coverage_percent', 'geometry']],
        name=label, show=visible,
        style_function=lambda feature, field=name: {
            'fillColor': colour_scale(feature['properties'][field]),
            'fillOpacity': 0.8, 'color': '#444444', 'weight': 0.6,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['region', name, 'coverage_percent'],
            aliases=['GeoCANOE region', 'Mean capacity factor (p.u.)', 'Region weather coverage (%)'],
            localize=True,
        ),
    ).add_to(interactive_preview)
folium.GeoJson(
    footprint_wgs, name='ERA5 weather footprint',
    style_function=lambda feature: {'fillOpacity': 0, 'color': '#b2182b',
                                    'weight': 2, 'dashArray': '6,4'},
).add_to(interactive_preview)
colour_scale.add_to(interactive_preview)
folium.LayerControl(collapsed=False).add_to(interactive_preview)
x0, y0, x1, y1 = footprint_wgs.total_bounds
interactive_preview.fit_bounds([[y0, x0], [y1, x1]])
interactive_preview.save(str(OUT / 'prov_quant_25km_capacity_factor_preview.html'))
display(interactive_preview)
print('Saved PNG and interactive HTML previews to:', OUT)

## 11. Export reproducible pilot artifacts
Files are written under ignored data directories. The manifest contains settings and versions, never credentials.


In [ ]:
profiles.to_netcdf(OUT / 'hourly_capacity_factors.nc')
sample_mean.to_dataframe().to_csv(OUT / 'sample_mean_by_weather_cell.csv')
hourly_diagnostic.to_csv(OUT / 'hourly_equal_cell_diagnostic.csv')
checks.to_csv(OUT / 'validation_summary.csv')
if regional_summary is not None:
    regional_summary.to_csv(OUT / 'regional_sample_mean_partial_coverage.csv')
manifest = {
    'created_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'configuration': CONFIG, 'versions': VERSIONS,
    'weather_dataset': 'reanalysis-era5-single-levels',
    'cutout_path': str(CUTOUT_PATH), 'hours': len(expected_time),
    'weather_cells': len(cutout.grid), 'timezone': 'UTC',
    'basemap_path': str(BASEMAP_PATH) if BASEMAP_PATH else None,
    'basemap_layer': BASEMAP_LAYER if BASEMAP_PATH else None,
    'interpretation': 'Sample-period resource profiles; no additional fleet losses; no annual claim.',
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved pilot artifacts:', OUT)


## 12. Interpretation and next experiments
- Check the kernel path, weather coverage, daytime PV shape and geographic patterns cell by cell.
- Repeat with a winter day, then a complete year and several weather years before selecting annual CFs.
- Validate against observed regional/plant generation and document losses and any bias correction.
- Compare turbines, hub heights, PV tilt/tracking and eligible-land/capacity allocation assumptions.
- Preserve coincident wind and solar hours when constructing representative periods.
- The current Gold builder has one season and one daily slice: it cannot represent hourly
  complementarity, renewable droughts or storage timing. Integration requires an explicit temporal design.
- No production schema or dependency lock files are changed by this notebook.


## Original wind research scope

# Notebook 20 — Wind Resource Representation for GeoCANOE

## 1. Objective

Develop and evaluate a reproducible workflow for representing spatially and temporally resolved wind electricity generation within GeoCANOE, using publicly accessible meteorological reanalysis and wind-resource datasets.

The workflow will investigate:

1. **Meteorological data acquisition:** Obtain hourly wind data from ERA5, with MERRA-2 as a potential comparison dataset.
2. **Wind energy conversion:** Use Atlite and specified turbine characteristics to estimate hourly electricity generation and capacity factors.
3. **Spatial representation:** Map meteorological data onto GeoCANOE's configurable geographic and projected basemaps.
4. **Spatial refinement:** Evaluate whether high-resolution Global Wind Atlas information can improve the geographical representation of wind resources.
5. **Model integration:** Establish reproducible outputs suitable for annual or representative-period capacity-factor constraints in the CANOE/TEMOA energy system optimization framework.

## 2. Initial methodological scope

- **Primary meteorological source:** ERA5 hourly reanalysis.
- **Alternative meteorological source:** NASA MERRA-2.
- **Energy conversion library:** Atlite.
- **High-resolution reference:** Global Wind Atlas.
- **Spatial model domain:** Canadian GeoCANOE regions.
- **Initial study area:** To be selected.
- **Initial study period:** To be selected.
- **Output:** Hourly and annual wind capacity factors, with spatial and temporal identifiers.

## 3. Methodological considerations

The analysis must distinguish between meteorological grid resolution, model-region resolution, turbine-specific generation characteristics, and the spatial and temporal aggregation methods used to construct representative wind profiles.

Particular attention will be given to:

- Wind-speed extrapolation to turbine hub height.
- Turbine power-curve assumptions.
- Temporal representativeness and interannual variability.
- Spatial interpolation versus physical downscaling.
- Geographic aggregation and potential spatial resource heterogeneity.
- Validation of derived capacity factors and uncertainty introduced by downscaling.

## 4. Integration boundary

This notebook is exploratory and does not modify GeoCANOE's production Silver or Gold workflows.

Validated methods may subsequently be incorporated into the `src/geocanoe/` package, with processed meteorological data and wind-generation profiles stored as reproducible intermediate products.
